# Migration Verification: EC_raw → metnorth

Verifies that the migration from `crmp` (EC_raw, network_id=19) to `metnorth` completed correctly.
Checks are run in dependency order: network → variables → stations → histories → observations.

In [8]:
import pandas as pd
import sqlalchemy as sa

crmp_url = "postgresql+psycopg2://crmp@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"
metnorth_url = "postgresql+psycopg2://metnorth@/metnorth?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"

crmp_engine = sa.create_engine(crmp_url, echo=False)
metnorth_engine = sa.create_engine(metnorth_url, echo=False)

CRMP_EC_RAW_NETWORK_ID = 19
NORTHERN_PROVINCES = ['NT', 'YT', 'NU']

def check(label, passed, detail=""):
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {label}" + (f"\n       {detail}" if detail else ""))
    return passed

## 1. Network

In [9]:
with crmp_engine.connect() as conn:
    crmp_network = conn.execute(sa.text("""
        SELECT network_name, description, virtual, publish, col_hex, network_display_name
        FROM meta_network WHERE network_id = :nid
    """), {"nid": CRMP_EC_RAW_NETWORK_ID}).mappings().fetchone()

with metnorth_engine.connect() as conn:
    metnorth_network = conn.execute(sa.text("""
        SELECT network_id, network_name, description, virtual, publish, col_hex, network_display_name
        FROM meta_network WHERE network_name = 'EC_raw'
    """)).mappings().fetchone()

check("EC_raw network exists in metnorth", metnorth_network is not None)

if metnorth_network:
    METNORTH_EC_RAW_NETWORK_ID = metnorth_network["network_id"]
    for col in ["network_name", "description", "virtual", "publish", "col_hex", "network_display_name"]:
        check(f"  network.{col} matches", crmp_network[col] == metnorth_network[col],
              f"crmp={crmp_network[col]!r}  metnorth={metnorth_network[col]!r}")

display(pd.DataFrame([dict(metnorth_network)]))

✅ PASS  EC_raw network exists in metnorth
✅ PASS    network.network_name matches
       crmp='EC_raw'  metnorth='EC_raw'
✅ PASS    network.description matches
       crmp='Environment and Climate Change Canada (raw observations from "Climate Data Online")'  metnorth='Environment and Climate Change Canada (raw observations from "Climate Data Online")'
✅ PASS    network.virtual matches
       crmp=None  metnorth=None
✅ PASS    network.publish matches
       crmp=True  metnorth=True
✅ PASS    network.col_hex matches
       crmp='#FF0000'  metnorth='#FF0000'
✅ PASS    network.network_display_name matches
       crmp='EC_Raw'  metnorth='EC_Raw'


,network_id,network_name,description,virtual,publish,col_hex,network_display_name
0,40,EC_raw,Environment and Climate Change Canada (raw obs...,None,True,#FF0000,EC_Raw


## 2. Variables

In [12]:
crmp_vars = pd.read_sql(sa.text("""
    SELECT net_var_name, standard_name, cell_method, unit, display_name, short_name
    FROM meta_vars WHERE network_id = :nid ORDER BY net_var_name
"""), crmp_engine, params={"nid": CRMP_EC_RAW_NETWORK_ID})

metnorth_vars = pd.read_sql(sa.text("""
    SELECT net_var_name, standard_name, cell_method, unit, display_name, short_name
    FROM meta_vars WHERE network_id = :nid ORDER BY net_var_name
"""), metnorth_engine, params={"nid": METNORTH_EC_RAW_NETWORK_ID})

check("Variable count matches",
      len(crmp_vars) == len(metnorth_vars),
      f"crmp={len(crmp_vars)}  metnorth={len(metnorth_vars)}")

# net_var_name is the unique identifier within a network — use it as the join key
merged_vars = crmp_vars.merge(metnorth_vars, on="net_var_name", suffixes=("_crmp", "_metnorth"))
check("All variables matched by net_var_name",
      len(merged_vars) == len(crmp_vars),
      f"{len(merged_vars)}/{len(crmp_vars)} matched")

missing = set(crmp_vars["net_var_name"]) - set(metnorth_vars["net_var_name"])
extra = set(metnorth_vars["net_var_name"]) - set(crmp_vars["net_var_name"])
if missing:
    print(f"       Missing from metnorth: {sorted(missing)}")
if extra:
    print(f"       Extra in metnorth:     {sorted(extra)}")

compare_cols = ["standard_name", "cell_method", "unit", "display_name", "short_name"]
for col in compare_cols:
    mismatches = merged_vars[merged_vars[f"{col}_crmp"] != merged_vars[f"{col}_metnorth"]]
    check(f"  vars.{col} values match", mismatches.empty, f"{len(mismatches)} mismatch(es)")

# Interleave crmp/metnorth columns so like values sit side-by-side
ordered_cols = ["net_var_name"] + [c for col in compare_cols for c in (f"{col}_crmp", f"{col}_metnorth")]
display_df = merged_vars[ordered_cols].copy()

# Highlight cells where the crmp/metnorth pair differs
def highlight_mismatches(row):
    styles = [""] * len(row)
    for col in compare_cols:
        crmp_idx = display_df.columns.get_loc(f"{col}_crmp")
        mn_idx = display_df.columns.get_loc(f"{col}_metnorth")
        if row.iloc[crmp_idx] != row.iloc[mn_idx]:
            styles[crmp_idx] = "background-color: #ffd6d6"
            styles[mn_idx] = "background-color: #ffd6d6"
    return styles

display(display_df.style.apply(highlight_mismatches, axis=1))

✅ PASS  Variable count matches
       crmp=19  metnorth=19
✅ PASS  All variables matched by net_var_name
       19/19 matched
✅ PASS    vars.standard_name values match
       0 mismatch(es)
✅ PASS    vars.cell_method values match
       0 mismatch(es)
✅ PASS    vars.unit values match
       0 mismatch(es)
✅ PASS    vars.display_name values match
       0 mismatch(es)
✅ PASS    vars.short_name values match
       0 mismatch(es)


,net_var_name,standard_name_crmp,standard_name_metnorth,cell_method_crmp,cell_method_metnorth,unit_crmp,unit_metnorth,display_name_crmp,display_name_metnorth,short_name_crmp,short_name_metnorth
0,air_temperature,air_temperature,air_temperature,time: point,time: point,Celsius,Celsius,Temperature (Point),Temperature (Point),air_temperature_point,air_temperature_point
1,air_temperature_yesterday_high,air_temperature,air_temperature,time: maximum,time: maximum,Celsius,Celsius,Temperature (Max.),Temperature (Max.),air_temperature_maximum,air_temperature_maximum
2,air_temperature_yesterday_low,air_temperature,air_temperature,time: minimum,time: minimum,Celsius,Celsius,Temperature (Min.),Temperature (Min.),air_temperature_minimum,air_temperature_minimum
3,dew_point,dew_point_temperature,dew_point_temperature,time: point,time: point,Celsius,Celsius,Dew Point Temperature (Point),Dew Point Temperature (Point),dew_point_temperature_point,dew_point_temperature_point
4,mean_sea_level,mean_sea_level_pressure,mean_sea_level_pressure,time: point,time: point,kPa,kPa,Mean Sea Level Pressure,Mean Sea Level Pressure,mean_seal_level_pressure_point,mean_seal_level_pressure_point
5,Precip_Climatology,lwe_thickness_of_precipitation_amount,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,t: sum within months t: mean over years,mm,mm,Precipitation Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum within months t: mean over years,lwe_thickness_of_precipitation_amountt: sum within months t: mean over years
6,rain_amount,thickness_of_rainfall_amount,thickness_of_rainfall_amount,time: sum,time: sum,mm,mm,Rainfall,Rainfall,thickness_of_rainfall_amount_sum,thickness_of_rainfall_amount_sum
7,relative_humidity,relative_humidity,relative_humidity,time: point,time: point,percent,percent,Relative Humidity (Point),Relative Humidity (Point),relative_humidity_point,relative_humidity_point
8,snow_amount,thickness_of_snowfall_amount,thickness_of_snowfall_amount,time: sum,time: sum,cm,cm,Snowfall Amount,Snowfall Amount,thickness_of_snowfall_amount_sum,thickness_of_snowfall_amount_sum
9,tendency_amount,tendency_of_air_pressure,tendency_of_air_pressure,time: sum,time: sum,kPa s-1,kPa s-1,Air Pressure Tendency,Air Pressure Tendency,tendency_of_air_pressure_sum,tendency_of_air_pressure_sum


## 3. Stations

In [13]:
crmp_stations = pd.read_sql(sa.text("""
    SELECT DISTINCT s.native_id, s.publish
    FROM meta_station s
    JOIN meta_history h ON s.station_id = h.station_id
    WHERE s.network_id = :nid
      AND h.province = ANY(:provinces)
    ORDER BY s.native_id
"""), crmp_engine, params={"nid": CRMP_EC_RAW_NETWORK_ID, "provinces": NORTHERN_PROVINCES})

metnorth_stations = pd.read_sql(sa.text("""
    SELECT native_id, publish
    FROM meta_station
    WHERE network_id = :nid
    ORDER BY native_id
"""), metnorth_engine, params={"nid": METNORTH_EC_RAW_NETWORK_ID})

check("Station count matches",
      len(crmp_stations) == len(metnorth_stations),
      f"crmp={len(crmp_stations)}  metnorth={len(metnorth_stations)}")

merged_stations = crmp_stations.merge(metnorth_stations, on="native_id", suffixes=("_crmp", "_metnorth"))
check("All stations matched by native_id",
      len(merged_stations) == len(crmp_stations),
      f"{len(merged_stations)}/{len(crmp_stations)} matched")

missing = set(crmp_stations["native_id"]) - set(metnorth_stations["native_id"])
extra = set(metnorth_stations["native_id"]) - set(crmp_stations["native_id"])
if missing:
    print(f"       Missing from metnorth: {sorted(missing)}")
if extra:
    print(f"       Extra in metnorth:     {sorted(extra)}")

display(metnorth_stations)

✅ PASS  Station count matches
       crmp=36  metnorth=36
✅ PASS  All stations matched by native_id
       36/36 matched


,native_id,publish
0,2100159,True
1,2100160,True
2,2100180,True
3,2100181,True
4,2100182,True
5,2100184,True
6,2100301,True
7,2100401,True
8,2100402,True
9,2100517,True


## 4. Histories

In [22]:

# COALESCE sentinels replace NULLs so pandas merge doesn't silently drop rows.
# Sentinel values are outside any realistic range for each field.
_hist_sql = """
    SELECT s.native_id, h.station_name, h.lon, h.lat, h.elev,
           h.sdate, h.edate, h.province, h.country, h.freq,
           COALESCE(h.lon,   999)          AS lon_key,
           COALESCE(h.lat,   999)          AS lat_key,
           COALESCE(h.elev,  -99999)       AS elev_key,
           COALESCE(h.sdate, '1900-01-01') AS sdate_key,
           COALESCE(h.edate, '9999-01-01') AS edate_key
    FROM meta_history h
    JOIN meta_station s ON h.station_id = s.station_id
    WHERE s.network_id = :nid {province_filter}
    ORDER BY s.native_id, lon_key, lat_key, elev_key, sdate_key
"""

crmp_histories = pd.read_sql(sa.text(
    _hist_sql.format(province_filter="AND h.province = ANY(:provinces)")
), crmp_engine, params={"nid": CRMP_EC_RAW_NETWORK_ID, "provinces": NORTHERN_PROVINCES})

metnorth_histories = pd.read_sql(sa.text(
    _hist_sql.format(province_filter="")
), metnorth_engine, params={"nid": METNORTH_EC_RAW_NETWORK_ID})

check("History count matches",
      len(crmp_histories) == len(metnorth_histories),
      f"crmp={len(crmp_histories)}  metnorth={len(metnorth_histories)}")

# Full location + date composite key — uniquely identifies each copied history row
key_cols = ["native_id", "lon_key", "lat_key", "elev_key", "sdate_key", "edate_key"]
merged_hist = crmp_histories.merge(metnorth_histories, on=key_cols, suffixes=("_crmp", "_metnorth"))
check("All histories matched by native_id + location + dates",
      len(merged_hist) == len(crmp_histories),
      f"{len(merged_hist)}/{len(crmp_histories)} matched")

crmp_keys = set(crmp_histories[key_cols].itertuples(index=False, name=None))
mn_keys = set(metnorth_histories[key_cols].itertuples(index=False, name=None))
missing = crmp_keys - mn_keys
extra = mn_keys - crmp_keys
if missing:
    print(f"       Missing from metnorth ({len(missing)}): {sorted(missing, key=lambda x: str(x))[:5]}")
if extra:
    print(f"       Extra in metnorth     ({len(extra)}): {sorted(extra, key=lambda x: str(x))[:5]}")

drop_cols = ["lon_key", "lat_key", "elev_key", "sdate_key", "edate_key"]
for col in ["station_name", "lon", "lat", "elev", "province", "country", "freq"]:
    mismatches = merged_hist[merged_hist[f"{col}_crmp"] != merged_hist[f"{col}_metnorth"]]
    # NaN != NaN in pandas but NULL == NULL semantically — exclude rows where both are null
    null_both = merged_hist[f"{col}_crmp"].isna() & merged_hist[f"{col}_metnorth"].isna()
    real_mismatches = mismatches[~null_both.loc[mismatches.index]]
    check(f"  history.{col} values match", real_mismatches.empty, f"{len(real_mismatches)} mismatch(es)")

display(metnorth_histories.drop(columns=drop_cols))


✅ PASS  History count matches
       crmp=60  metnorth=60
✅ PASS  All histories matched by native_id + location + dates
       60/60 matched
✅ PASS    history.station_name values match
       0 mismatch(es)
✅ PASS    history.lon values match
       0 mismatch(es)
✅ PASS    history.lat values match
       0 mismatch(es)
✅ PASS    history.elev values match
       0 mismatch(es)
✅ PASS    history.province values match
       0 mismatch(es)
✅ PASS    history.country values match
       0 mismatch(es)
✅ PASS    history.freq values match
       0 mismatch(es)


,native_id,station_name,lon,lat,elev,sdate,edate,province,country,freq
0,2100159,Beaver Creek Airport,-140.868889,62.410278,649.50,2014-12-04,None,YT,None,1-hourly
1,2100160,Beaver Creek Airport,-140.867500,62.410278,649.00,2012-01-24,None,YT,None,1-hourly
2,2100180,Burwash Airport,-139.040000,61.370000,806.20,2013-01-15,None,YT,None,1-hourly
3,2100181,Burwash Airport,-139.040000,61.370556,805.30,2012-04-02,None,YT,None,1-hourly
4,2100181,Burwash Airport,-139.032234,61.370008,805.30,2012-01-24,2012-02-28,YT,None,1-hourly
5,2100182,Burwash Airport,-139.050000,61.366667,806.20,2012-02-28,None,YT,None,1-hourly
6,2100184,Burwash,-139.016667,61.366667,807.00,2013-11-14,None,YT,None,1-hourly
7,2100301,Carmacks,-136.191972,62.115000,542.90,2012-01-24,None,YT,None,1-hourly
8,2100401,Dawson Airport,-139.130278,64.042222,370.30,2013-10-10,None,YT,None,1-hourly
9,2100402,Dawson Airport,-139.127778,64.043056,370.33,2012-01-24,None,YT,None,1-hourly


## 5. Observation Counts per History

In [8]:
crmp_obs_counts = pd.read_sql(sa.text("""
    SELECT s.native_id, h.sdate, h.edate, COUNT(*) AS obs_count
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.network_id = :nid
      AND h.province = ANY(:provinces)
      AND v.network_id = :nid
    GROUP BY s.native_id, h.sdate, h.edate
    ORDER BY s.native_id, h.sdate
"""), crmp_engine, params={"nid": CRMP_EC_RAW_NETWORK_ID, "provinces": NORTHERN_PROVINCES})

metnorth_obs_counts = pd.read_sql(sa.text("""
    SELECT s.native_id, h.sdate, h.edate, COUNT(*) AS obs_count
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.network_id = :nid
      AND v.network_id = :nid
    GROUP BY s.native_id, h.sdate, h.edate
    ORDER BY s.native_id, h.sdate
"""), metnorth_engine, params={"nid": METNORTH_EC_RAW_NETWORK_ID})

total_crmp = crmp_obs_counts["obs_count"].sum()
total_metnorth = metnorth_obs_counts["obs_count"].sum()
check("Total observation count matches",
      total_crmp == total_metnorth,
      f"crmp={total_crmp:,}  metnorth={total_metnorth:,}")

merged_counts = crmp_obs_counts.merge(
    metnorth_obs_counts, on=["native_id", "sdate", "edate"], suffixes=("_crmp", "_metnorth")
)
count_mismatches = merged_counts[merged_counts["obs_count_crmp"] != merged_counts["obs_count_metnorth"]]
check("Per-history observation counts match",
      count_mismatches.empty,
      f"{len(count_mismatches)} histories with differing counts")

if not count_mismatches.empty:
    display(count_mismatches)
else:
    display(merged_counts)

✅ PASS  Total observation count matches
       crmp=20,019,148  metnorth=20,019,148
✅ PASS  Per-history observation counts match
       0 histories with differing counts


,native_id,sdate,edate,obs_count_crmp,obs_count_metnorth
0,2100159,2014-12-04,None,661743,661743
1,2100160,2012-01-24,None,57029,57029
2,2100180,2013-01-15,None,847755,847755
3,2100181,2012-01-24,2012-02-28,6050,6050
4,2100181,2012-04-02,None,51649,51649
5,2100182,2012-02-28,None,5566,5566
6,2100184,2013-11-14,None,731915,731915
7,2100301,2012-01-24,None,774873,774873
8,2100401,2013-10-10,None,782456,782456
9,2100402,2012-01-24,None,77912,77912


## 6. Sample Data Spot-check

Pick one station per province and compare a sample of obs_raw values by `obs_time` and `vars_id` (matched by `standard_name + cell_method`).

In [25]:

SAMPLE_SIZE = 1000

for province in NORTHERN_PROVINCES:
    # Pick the first crmp history in this province by native_id
    with crmp_engine.connect() as conn:
        sample_row = conn.execute(sa.text("""
            SELECT s.native_id, h.history_id, h.lon, h.lat, h.elev, h.sdate, h.edate
            FROM meta_history h
            JOIN meta_station s ON h.station_id = s.station_id
            WHERE s.network_id = :nid AND h.province = :province
            ORDER BY s.native_id LIMIT 1
        """), {"nid": CRMP_EC_RAW_NETWORK_ID, "province": province}).fetchone()

    if sample_row is None:
        print(f"⚠️  No stations found in {province}, skipping")
        continue

    native_id, crmp_history_id, lon, lat, elev, sdate, edate = sample_row

    crmp_sample = pd.read_sql(sa.text("""
        SELECT o.obs_time, o.datum, v.standard_name, v.cell_method
        FROM obs_raw o
        JOIN meta_vars v ON o.vars_id = v.vars_id
        WHERE o.history_id = :hid
        ORDER BY o.obs_time
        LIMIT :n
    """), crmp_engine, params={"hid": int(crmp_history_id), "n": SAMPLE_SIZE})

    # Look up the corresponding metnorth history by matching the copied field values
    with metnorth_engine.connect() as conn:
        mn_history_id = conn.execute(sa.text("""
            SELECT h.history_id
            FROM meta_history h
            JOIN meta_station s ON h.station_id = s.station_id
            WHERE s.network_id = :nid
              AND s.native_id  = :native_id
              AND h.lon   IS NOT DISTINCT FROM :lon
              AND h.lat   IS NOT DISTINCT FROM :lat
              AND h.elev  IS NOT DISTINCT FROM :elev
              AND h.sdate IS NOT DISTINCT FROM :sdate
              AND h.edate IS NOT DISTINCT FROM :edate
        """), {"nid": METNORTH_EC_RAW_NETWORK_ID, "native_id": native_id,
               "lon": lon, "lat": lat, "elev": elev, "sdate": sdate, "edate": edate}).scalar()

    if mn_history_id is None:
        check(f"[{province}] {native_id}: metnorth history found", False)
        continue

    metnorth_sample = pd.read_sql(sa.text("""
        SELECT o.obs_time, o.datum, v.standard_name, v.cell_method
        FROM obs_raw o
        JOIN meta_vars v ON o.vars_id = v.vars_id
        WHERE o.history_id = :hid
        ORDER BY o.obs_time
        LIMIT :n
    """), metnorth_engine, params={"hid": int(mn_history_id), "n": SAMPLE_SIZE})

    merged_sample = crmp_sample.merge(
        metnorth_sample, on=["obs_time", "standard_name", "cell_method"], suffixes=("_crmp", "_metnorth")
    )
    datum_mismatches = merged_sample[merged_sample["datum_crmp"] != merged_sample["datum_metnorth"]]
    check(f"[{province}] {native_id}: {len(crmp_sample)} sample rows — datum values match",
          datum_mismatches.empty,
          f"{len(datum_mismatches)} mismatch(es) in sample")


⚠️  No stations found in NT, skipping
✅ PASS  [YT] 2100159: 1000 sample rows — datum values match
       0 mismatch(es) in sample
⚠️  No stations found in NU, skipping
